# PE6201 · A2 — scaffold tour · **PROBLEM B**

## What this notebook is

**One case, followed end to end, with real values printed at every step.**

The case is **`REF-5602`** — a Problem B referral, and the booking worked example in
Appendix A of the brief. Every cell below uses that one referral. Wherever you see a
hard-coded value — `'REF-5602'`, `'OPH'`, `'P-1180'`, a date — it came from that
record, and the markdown says where.

**The model is simulated, the data is real.** The backend replays a fixed sequence of
moves (`SCRIPTS['REF-5602']` in `backends.py`) so the run is deterministic and free.
The tools underneath are doing genuine lookups against the shipped JSON — nothing here
is faked except the model's decisions.


> **Doing Problem A?** Use `A2_Scaffold_Tour_ProblemA.ipynb` instead. Same nine steps,
> same scaffold, different case. You only need the one for the problem you chose.

## What this notebook is not

**It contains no logic of its own.** Every cell imports from the `.py` files beside it.
That is deliberate, and it is the habit to copy:

> **Notebooks explore. Modules ship.**

Six people can edit six modules at once. Six people editing one notebook produces merge
conflicts and an unreadable diff — and section 8 of the brief leans on your commit
history to corroborate who did what.

**What you submit is the modules and `run_eval.py`, not this notebook.** D5(a) says a
marker clones your repository and runs it. `python3 run_eval.py` is that; a notebook
is not.

---
### Running this in Colab
Upload `A2_scaffold/` and `A2_reference_data/` to the same folder in your Drive, then
edit the two paths in the setup cell. Running locally: just run the cells in order.


## Cell 1 · Setup

Puts the scaffold on the import path and prints which backend is active.

**Read the printed line.** `BACKEND=scripted` is what makes this run free, offline and
identical every time. It is also what must be the committed default in your submission
— if a marker's clone needs a key, D5(a) has failed.

**If the printed line disagrees with `config.py`**, the scaffold will say so in capitals — Python is reusing cached bytecode. Restart the kernel and re-run.


In [ ]:
# --- SETUP ---------------------------------------------------------
# Local: this cell works as-is. Colab: set the two paths below.
import os, sys, json

SCAFFOLD = '.'          # e.g. '/content/drive/MyDrive/PE6201/A2_scaffold'
# os.environ['A2_DATA'] = '/content/drive/MyDrive/PE6201/A2_reference_data'

sys.path.insert(0, SCAFFOLD)
import config
print(config.summary())
print('data:', config.data_root())


---
## Cell 2 · What the agent is handed

The harness gives the agent **one referral id and nothing else**. This cell fetches
`REF-5602` so you can see exactly what that record contains.

**Look at what is *not* in it.** No urgency band. No red-flag verdict. No slot. No rule.
Every one of those has to be fetched by a tool during the run — which is what makes
this an agent loop rather than one big prompt.

**Two fields to note, because the next cells use them:**

- `specialty` is **`OPH`** (Ophthalmology). That is why every later cell passes `'OPH'`
  — it is not a choice we made, it is what this referral says.
- `patient_id` is **`P-1180`**, used in the duplicate check.


In [ ]:
import tools

referral = tools.get_referral('REF-5602')
print(json.dumps(referral, indent=1))
print()
print('specialty on this referral :', referral['specialty'])
print('patient on this referral   :', referral['patient_id'])


---
## Cell 3 · The rules the agent must reach through its tools

These facts live in `data_B/specialties.json`, `urgency_bands.json` and `as_of.json`.
**The agent never sees those files.** It asks a tool a question and gets one answer.
An agent handed all of this in its first prompt is making a single call, not running a
loop — exactly what D0(a) asks you to defend.

`check_referral_criteria('OPH', 'REF-5602')` — **`'OPH'` because that is this
referral's specialty**, from cell 2 — runs the department's protocol against the GP's
free text and reports five facts.

**It decides nothing.** The agent decides what the five facts mean, and the order
matters: a red flag ends the run, then a wrong department, then a missing test. Only
if all of those pass does a slot ever get queried.

For this referral: no red flag, right department, no missing tests, band **`routine`**,
window **8 weeks**. So the run continues — and the 8-week window from `as_of` is what
cell 4 uses.


In [ ]:
print('the clock (as_of):', tools.as_of())
print()
criteria = tools.check_referral_criteria('OPH', 'REF-5602')   # OPH: see cell 2
print(json.dumps(criteria, indent=1))
print()
print('-> band %r means %s %d-week window measured from as_of'
      % (criteria['band'],
         'an' if criteria['window_weeks'] == 8 else 'a',
         criteria['window_weeks']))


---
## Cell 4 · The trap in this case

The band is `routine`, so the window is 8 weeks from `as_of` 2026-09-09 — closing
**2026-11-04**.

This cell lists **every OPH slot in the data** and marks the ones that are inside that
window and have capacity free, **but are in the wrong band**. There are three: two
urgent and one soon, all of them *earlier* than the correct answer.

**A team filtering slots by date alone books one of these and fails the case.** Only
the band excludes them.

That is why `get_clinic_slots` takes `band` as a **required argument** rather than an
optional filter. Poka-yoke: make the wrong call impossible, not merely documented.

*(This cell reaches into `tools._load` to show you the raw table. That is a teaching
shortcut — your agent must never do it.)*


In [ ]:
WINDOW_OPEN, WINDOW_CLOSE = '2026-09-09', '2026-11-04'   # routine, 8 weeks from as_of

for s in tools._load('B', 'clinic_slots'):
    if s['specialty'] != 'OPH':
        continue
    inside = WINDOW_OPEN <= s['date'] <= WINDOW_CLOSE
    free   = s['capacity_remaining'] > 0
    if inside and free and s['band'] != 'routine':
        note = '   <-- in window, free, but WRONG BAND'
    elif inside and free:
        note = '   <-- legal and bookable'
    elif inside and not free:
        note = '   (in window but FULL - exists, cannot be booked)'
    else:
        note = ''
    print('%-8s %-8s %s %s  cap=%d%s'
          % (s['clinic'], s['band'], s['date'], s['time'],
             s['capacity_remaining'], note))


---
## Cell 5 · The run itself

Now the whole thing, turn by turn. `verbose=True` prints the model's thought and every
tool result.

**Four turns, six tool calls** — and this matches Appendix A exactly, which is worth
checking rather than taking on trust:

| Turn | Calls | Why grouped this way |
|---|---|---|
| 1 | `get_referral` | must run alone — everything else needs what it returns |
| 2 | `check_referral_criteria` + `lookup_patient` | independent of each other |
| 3 | `get_clinic_slots` × 2 | both halves of the window at once |
| 4 | `book_slot` | the gated action — a turn like any other |

**Turn 3 is a gamble worth noticing.** Firing both slot queries together makes this
four turns instead of five — but if the near window had held a slot, the second query
was wasted. That is the parallel-versus-sequential trade in one line, and it is what
D2(c) asks you to reason about.

The `thought` text is scripted, not generated. It is there to show you what a model
*would* be reasoning at each step.


In [ ]:
from agent import run_case

record = run_case('REF-5602', verbose=True)


---
## Cell 6 · The decision record

This is what gets graded. Two halves worth separating:

**The answer** — `decision`, `booked`, `reason`. This is what the routing table in
Appendix A cares about.

**The instrumentation** — `turns`, `tokens_in`, `cost_usd`, `evidence`,
`guardrails_fired`. Captured *while the run happened*, because you cannot report a
failure you had no way of noticing. D6's cost model and D7's loop failure both need
these, and a team that adds instrumentation afterwards re-runs the whole battery.

Note `guardrails_fired` shows `gate_passed` for `book_slot` — the irreversible step
went through the autonomy gate, and the record proves it.

**The token numbers are estimates**, because the scripted backend has no model. They
are here so your cost arithmetic has something to chew on. D6 wants *measured* counts,
which means the live battery.


In [ ]:
print(json.dumps(record, indent=2))


---
## Cell 7 · Grading — the code check

Deterministic comparison against the answer key. **No model, no person, no opinion.**
This is what produces the number.

The key row for `REF-5602` is printed first so you can see what is being compared:
the `expected_decision`, and — because this is a booking — the exact clinic, date and
time in `booked`.

What is **not** compared: the wording, the turn count, the cost. Two agents can both
be right and cost very different amounts, which is the subject of D6.


In [ ]:
from harness import load_key, code_check, prepare_judgement_check

expected = load_key('B')['REF-5602']
print('THE ANSWER KEY SAYS:')
print(json.dumps(expected, indent=1))

passed, fails = code_check(record, expected)
print()
print('CODE CHECK:', 'PASS' if passed else 'FAIL')
for f in fails:
    print('   ', f)


---
## Cell 8 · Grading — the judgement check

**This is the half a pass rate cannot show you.** With three possible outcomes a
coin-flip scores 33% on the code check alone, and an agent can reach the right
decision for the wrong reason without the code check noticing.

`must_record` lists what a full-marks record has to carry for this case. Those items
are written in English — a substring match would be theatre, not a check. So this
function **builds a queue, it does not decide**. Someone reads the reason and rules on
each item: a person can do it, or a second model can. Same kind of check; the only
difference is who grades.

Note `verdict` and `graded_by` come back empty. **Filling them in is your job**, and
if you use a model to do it, say so in the report — a model grading a model is a claim
that needs defending.


In [ ]:
item = prepare_judgement_check(record, expected)

print('The reason the agent gave:')
print('   ', item['reason'])
print()
print('Does it carry each of these? Nobody has ruled yet:')
for m in item['must_record']:
    print('  [ ]', m)
print()
print('verdict:', item['verdict'], '   graded_by:', item['graded_by'])


---
## Cell 9 · The failure that raises no exception  (D7)

Same case, same data. **One guard deleted** — action de-duplication — and nothing else
changed. That is the shape D7 requires: *the working agent, minus X*. Putting X back
recovers the behaviour, which is what makes it a diagnosis rather than a story.

Watch the three numbers in the BEFORE and AFTER lines:

- **turns** 4 → 6
- **cost** roughly 1.8× higher
- **decision** unchanged — still `book`, still the right slot

**No exception. No error. The right answer.** A pass-rate table would show this run as
a clean pass. And neither the step cap (8 turns) nor the budget ceiling (60,000 tokens)
fires, because neither is breached — they bound the damage, they do not detect the
fault.

You only ever see this **if you are counting**. That is the whole lesson of D7, and the
reason instrumentation is a requirement rather than a nicety.


In [ ]:
import demo_loop_failure
demo_loop_failure.main()


---
## Cell 10 · Now make it yours

Everything above ran on **one scripted case**. Your set needs 30–50. Here is the order
that works:

1. **`config.py`** — set `PROBLEM` to the one you chose. If it is A, re-run this
   notebook against `CLM-8842`, which is scripted the same way.
2. **`tools.py`** — read the comment block on every tool, then read the two six-field
   descriptors at the bottom. Now write deliberately worse ones: that is your D2(b)
   **v1**, and the measured comparison is the deliverable.
3. **`backends.py`** — script a second case yourself. **If you cannot write the steps
   down, you do not yet understand the case.** Better to find that out now than at 2am
   on the 13th.
4. **`expected_outcomes_*.json`** — label every case you add, **from Appendix A's
   routing table, before you run the agent on it**. See
   `PE6201_A2_Adding_Extra_Cases.pdf`.
5. **`guardrails.py`** — set the limits from evidence. If your median run is 4 turns
   and your worst legitimate run is 7, a cap of 8 is defensible and a cap of 30 is
   decoration.

Then close this notebook and work in the modules.

```bash
python3 run_eval.py
```
